In [ ]:
import json
import pandas as pd
import numpy as np
import glob
import os
import re

metadata_files = glob.glob("*.json")
gps_samples, ypr_samples = [], []

for file in metadata_files:
    with open(file, "r") as f:
        data = json.load(f)

    for stream_id, stream_data in data.items():
        if isinstance(stream_data, dict) and "streams" in stream_data:
            streams = stream_data["streams"]

            # === GPS ===
            gps_stream = streams.get("GPS5")
            if gps_stream and "samples" in gps_stream:
                for s in gps_stream["samples"]:
                    val = s.get("value", [])
                    if len(val) >= 3:
                        gps_samples.append({
                            "file": file,
                            "stream_id": stream_id,
                            "cts": s.get("cts"),
                            "datetime": s.get("date"),
                            "lat": val[0],
                            "lon": val[1],
                            "alt": val[2] if len(val) > 2 else None,
                            "spd_3d": val[3] if len(val) > 3 else None,
                            "spd_2d": val[4] if len(val) > 4 else None
                        })

            # === Yaw / Pitch / Roll ===
            for key in ["CORI", "GYRO", "ISOG", "CAMERA_ORIENTATION"]:
                if key in streams:
                    ypr_stream = streams[key]
                    if ypr_stream and "samples" in ypr_stream:
                        for s in ypr_stream["samples"]:
                            val = s.get("value", [])
                            if len(val) >= 3:
                                ypr_samples.append({
                                    "file": file,
                                    "stream_id": stream_id,
                                    "cts": s.get("cts"),
                                    "datetime": s.get("date"),
                                    "yaw": val[0],
                                    "pitch": val[1],
                                    "roll": val[2]
                                })

# === Buat DataFrame ===
gps_df = pd.DataFrame(gps_samples)
ypr_df = pd.DataFrame(ypr_samples)

# --- convert datetime ---
gps_df["datetime"] = pd.to_datetime(gps_df["datetime"], errors="coerce")
ypr_df["datetime"] = pd.to_datetime(ypr_df["datetime"], errors="coerce")

# --- waktu relatif (detik dari awal GPS) ---
if not gps_df.empty:
    start_time = gps_df["datetime"].min()
    gps_df["t_rel_sec"] = (gps_df["datetime"] - start_time).dt.total_seconds()

if not ypr_df.empty:
    ypr_df["t_rel_sec"] = (ypr_df["datetime"] - start_time).dt.total_seconds()

# --- interpolasi YPR ke timeline GPS ---
ypr_interp = {}
for col in ["yaw", "pitch", "roll"]:
    if not ypr_df.empty:
        ypr_interp[col] = np.interp(
            gps_df["t_rel_sec"],
            ypr_df["t_rel_sec"],
            ypr_df[col]
        )
    else:
        ypr_interp[col] = np.nan

for k, v in ypr_interp.items():
    gps_df[k] = v

# === Tambah placeholder sensor lain (biar seragam) ===
for col in ["gyro_x","gyro_y","gyro_z","accel_x","accel_y","accel_z","temp"]:
    gps_df[col] = np.nan

# === Simpan hasil gabungan ===
gps_df.to_csv("gps_ypr_merged.csv", index=False)
print(f" Gabungan selesai — total {len(gps_df)} baris, disimpan ke gps_ypr_merged.csv")
print(gps_df.head(10))


In [ ]:
import os
import re
import pandas as pd
import numpy as np

# === PARAMETER ===
frames_dir = "input"  # folder gambar
merged_sensor_csv = "gps_ypr_merged.csv"
output_csv = "frames_mapping_fullsensor_with_ypr.csv"

# === LOAD DATA SENSOR ===
gps_df = pd.read_csv(merged_sensor_csv)
gps_df = gps_df.sort_values("t_rel_sec").reset_index(drop=True)

t_min, t_max = gps_df["t_rel_sec"].min(), gps_df["t_rel_sec"].max()

# === DETEKSI FRAME INPUT ===
img_files = sorted(
    [f for f in os.listdir(frames_dir) if re.match(r"^GS\d+_image_\d+_\d+\.jpg", f)]
)
if len(img_files) == 0:
    raise SystemExit("⚠️ Tidak ditemukan file gambar dengan pola image_xxx_x.jpg")

print(f"Total file cocok pola regex: {len(img_files)}")

# === GROUP BERDASARKAN ID UTAMA (misal image_0177_xx) ===
grouped = {}
for fname in img_files:
    m = re.match(r"^GS\d+_image_(\d+)_\d+\.jpg", fname)
    if m:
        group_id = int(m.group(1))
        grouped.setdefault(group_id, []).append(fname)

group_ids = sorted(grouped.keys())
print(f"Total grup frame unik: {len(group_ids)}")

# === DURASI VIDEO & TIMELINE ===
video_duration = 59.0  # ubah sesuai kebutuhan
timestamps = np.linspace(0, video_duration, len(group_ids))

# === INTERPOLASI SENSOR PER GROUP ===
mapping_rows = []
for idx, (gid, t_vid) in enumerate(zip(group_ids, timestamps)):
    t_clipped = np.clip(t_vid, t_min, t_max)

    # Interpolasi data dasar (GPS dan orientasi utama)
    interpolated = {}
    for col in ["lat","lon","alt","spd_3d","spd_2d",
                "yaw","pitch","roll",
                "gyro_x","gyro_y","gyro_z",
                "accel_x","accel_y","accel_z","temp"]:
        if col in gps_df.columns:
            interpolated[col] = np.interp(t_clipped, gps_df["t_rel_sec"], gps_df[col])
        else:
            interpolated[col] = np.nan

    # === Tiap grup punya 8 frame (0–7) ===
    subfiles = sorted(grouped[gid], key=lambda x: int(x.split("_")[-1].split(".")[0]))
    n_sub = len(subfiles)
    base_yaw = interpolated["yaw"]

    for i, subf in enumerate(subfiles):
        step = 360.0 / n_sub
        sub_yaw = base_yaw + (i * step)  # rotasi yaw per 45 derajat (360/8)
        sub_yaw = (sub_yaw + 360) % 360  # normalisasi yaw ke [0, 360)
        row = {
            "image_name": subf,
            "timestamp_video": t_vid,
            "lat": interpolated["lat"],
            "lon": interpolated["lon"],
            "alt": interpolated["alt"],
            "spd_3d": interpolated["spd_3d"],
            "spd_2d": interpolated["spd_2d"],
            "yaw": sub_yaw,
            "pitch": interpolated["pitch"],
            "roll": interpolated["roll"],
            "gyro_x": interpolated["gyro_x"],
            "gyro_y": interpolated["gyro_y"],
            "gyro_z": interpolated["gyro_z"],
            "accel_x": interpolated["accel_x"],
            "accel_y": interpolated["accel_y"],
            "accel_z": interpolated["accel_z"],
            "temp": interpolated["temp"],
        }
        mapping_rows.append(row)

# === Simpan ke CSV ===
mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(output_csv, index=False)

print(f"\nMapping selesai — total {len(mapping_df)} baris disimpan ke {output_csv}")
print(f"Total group GPS : {len(group_ids)}")
print(f"Rentang waktu video : 0.00 → {video_duration:.2f} detik")
print(f"Rentang waktu GPS : {t_min:.2f} → {t_max:.2f} detik\n")

print("--- Validasi ---")
print(f"Total gambar di folder input : {len(img_files)}")
print(f"Total baris di CSV mapping   : {len(mapping_df)}")
if len(img_files) == len(mapping_df):
    print(" Semua gambar berhasil terpetakan ke metadata GPS (1 koordinat per 8 potongan).")

print("\nContoh 8 baris pertama:")
print(mapping_df.head(8))


Total file cocok pola regex: 1424
Total grup frame unik: 178

✅ Mapping selesai — total 1424 baris disimpan ke frames_mapping_fullsensor_with_ypr.csv
Total group GPS : 178
Rentang waktu video : 0.00 → 59.00 detik
Rentang waktu GPS : 0.00 → 60.39 detik

--- Validasi ---
Total gambar di folder input : 1424
Total baris di CSV mapping   : 1424
✅ Semua gambar berhasil terpetakan ke metadata GPS (1 koordinat per 8 potongan).

Contoh 8 baris pertama:
         image_name  timestamp_video       lat         lon      alt  spd_3d  \
0  image_0001_0.jpg              0.0 -6.926128  107.637465  708.407    0.07   
1  image_0001_1.jpg              0.0 -6.926128  107.637465  708.407    0.07   
2  image_0001_2.jpg              0.0 -6.926128  107.637465  708.407    0.07   
3  image_0001_3.jpg              0.0 -6.926128  107.637465  708.407    0.07   
4  image_0001_4.jpg              0.0 -6.926128  107.637465  708.407    0.07   
5  image_0001_5.jpg              0.0 -6.926128  107.637465  708.407    0.07   

In [ ]:
import piexif
from PIL import Image
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from subprocess import run

# === Path CSV & Output ===
csv_path = "frames_mapping_fullsensor_with_ypr.csv"
frames_dir = "input"  # folder tempat gambar asli
output_dir = "./output_with_full_exif3"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)

def deg_to_dms_rational(deg_float):
    """Konversi derajat desimal ke format DMS untuk EXIF GPS"""
    deg = int(deg_float)
    min_float = abs(deg_float - deg) * 60
    minutes = int(min_float)
    sec = round((min_float - minutes) * 60 * 100)
    return ((abs(deg), 1), (minutes, 1), (sec, 100))


for _, row in df.iterrows():
    img_name = row.get("image_name", None)
    if not img_name:
        print("Skip: baris tanpa image_name.")
        continue

    img_path = os.path.join(frames_dir, img_name)
    if not os.path.exists(img_path):
        print(f"Skip: {img_path} tidak ditemukan.")
        continue

    # === Buka gambar & buat dict EXIF ===
    img = Image.open(img_path)
    exif_dict = {"0th": {}, "Exif": {}, "GPS": {}, "1st": {}, "thumbnail": None}

    # === GPS info ===
    lat, lon, alt = row["lat"], row["lon"], row["alt"]
    exif_dict["GPS"][piexif.GPSIFD.GPSLatitudeRef] = "S" if lat < 0 else "N"
    exif_dict["GPS"][piexif.GPSIFD.GPSLatitude] = deg_to_dms_rational(lat)
    exif_dict["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
    exif_dict["GPS"][piexif.GPSIFD.GPSLongitude] = deg_to_dms_rational(lon)
    exif_dict["GPS"][piexif.GPSIFD.GPSAltitude] = (int(alt * 100), 100)

    # === Datetime ===
    try:
        dt_obj = datetime.fromisoformat(str(row.get("datetime", "")))
        formatted_time = dt_obj.strftime("%Y:%m:%d %H:%M:%S")
        exif_dict["0th"][piexif.ImageIFD.DateTime] = formatted_time
        exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal] = formatted_time
    except Exception:
        pass

    # === Heading (Yaw) ===
    yaw_deg = row.get("yaw", np.nan)
    if not pd.isna(yaw_deg):
        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirectionRef] = "T"
        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirection] = (int(yaw_deg * 100), 100)

    # === Pitch & Roll ===
    pitch_deg = row.get("pitch", np.nan)
    roll_deg = row.get("roll", np.nan)

    # === Metadata tambahan (JSON di ImageDescription) ===
    extra_cols = [
        "gyro_x", "gyro_y", "gyro_z",
        "accel_x", "accel_y", "accel_z",
        "spd_3d", "spd_2d", "temp"
    ]
    metadata_json = {k: row[k] for k in extra_cols if k in row}
    metadata_json.update({
        "yaw_deg": yaw_deg,
        "pitch_deg": pitch_deg,
        "roll_deg": roll_deg
    })
    exif_dict["0th"][piexif.ImageIFD.ImageDescription] = json.dumps(metadata_json).encode("utf-8")

    # === Simpan output dengan EXIF dasar ===
    out_path = os.path.join(output_dir, img_name)
    img.save(out_path, exif=piexif.dump(exif_dict))
    print(f"📸 {img_name} → EXIF dasar lengkap (GPS + yaw + JSON)")

    # === Inject XMP pose (universal) ===
    exiftool_args = ["exiftool", "-overwrite_original"]
    if not pd.isna(yaw_deg):
        exiftool_args.append(f"-XMP:PoseYawDegrees={yaw_deg}")
    if not pd.isna(pitch_deg):
        exiftool_args.append(f"-XMP:PosePitchDegrees={pitch_deg}")
    if not pd.isna(roll_deg):
        exiftool_args.append(f"-XMP:PoseRollDegrees={roll_deg}")
    exiftool_args.append(out_path)
    run(exiftool_args)

    print(f"{img_name} → XMP PoseYaw/Pitch/Roll ditambahkan\n")

print("Selesai! Semua gambar sudah punya EXIF + XMP lengkap.")


📸 image_0001_0.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_0.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_1.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_1.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_2.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_2.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_3.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_3.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_4.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_4.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_5.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_5.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_6.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_6.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0001_7.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0001_7.jpg → XMP PoseYaw/Pitch/Roll ditambahkan

📸 image_0002_0.jpg → EXIF dasar lengkap (GPS + yaw + JSON)
✅ image_0002_

KeyboardInterrupt: 

In [ ]:
import piexif
from PIL import Image
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from subprocess import run

# === Path CSV & Output ===
csv_path = "frames_mapping_fullsensor_with_ypr.csv"
frames_dir = "input"  # folder tempat gambar asli
output_dir = "./output_with_full_exif3"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)

def deg_to_dms_rational(deg_float):
    """Konversi derajat desimal ke format DMS untuk EXIF GPS"""
    deg = int(deg_float)
    min_float = abs(deg_float - deg) * 60
    minutes = int(min_float)
    sec = round((min_float - minutes) * 60 * 100)
    return ((abs(deg), 1), (minutes, 1), (sec, 100))


for _, row in df.iterrows():
    img_name = row.get("image_name", None)
    if not img_name:
        print("⚠️  Skip: baris tanpa image_name.")
        continue

    img_path = os.path.join(frames_dir, img_name)
    if not os.path.exists(img_path):
        print(f"⚠️  Skip: {img_path} tidak ditemukan.")
        continue

    # === Buka gambar & buat dict EXIF ===
    img = Image.open(img_path)
    exif_dict = {"0th": {}, "Exif": {}, "GPS": {}, "1st": {}, "thumbnail": None}

    # === GPS info (tanpa altitude) ===
    lat, lon = row["lat"], row["lon"]
    exif_dict["GPS"][piexif.GPSIFD.GPSLatitudeRef] = "S" if lat < 0 else "N"
    exif_dict["GPS"][piexif.GPSIFD.GPSLatitude] = deg_to_dms_rational(lat)
    exif_dict["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
    exif_dict["GPS"][piexif.GPSIFD.GPSLongitude] = deg_to_dms_rational(lon)

    # === Datetime ===
    try:
        dt_obj = datetime.fromisoformat(str(row.get("datetime", "")))
        formatted_time = dt_obj.strftime("%Y:%m:%d %H:%M:%S")
        exif_dict["0th"][piexif.ImageIFD.DateTime] = formatted_time
        exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal] = formatted_time
    except Exception:
        pass

    # === Heading (Yaw) ===
    yaw_deg = row.get("yaw", np.nan)
    if not pd.isna(yaw_deg):
        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirectionRef] = "T"
        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirection] = (int(yaw_deg * 100), 100)

    # === Pitch & Roll ===
    pitch_deg = row.get("pitch", np.nan)
    roll_deg = row.get("roll", np.nan)

    # === Metadata tambahan (JSON di ImageDescription) ===
    extra_cols = [
        "gyro_x", "gyro_y", "gyro_z",
        "accel_x", "accel_y", "accel_z",
        "spd_3d", "spd_2d", "temp"
    ]
    metadata_json = {k: row[k] for k in extra_cols if k in row}
    metadata_json.update({
        "yaw_deg": yaw_deg,
        "pitch_deg": pitch_deg,
        "roll_deg": roll_deg
    })
    exif_dict["0th"][piexif.ImageIFD.ImageDescription] = json.dumps(metadata_json).encode("utf-8")

    # === Simpan output dengan EXIF dasar ===
    out_path = os.path.join(output_dir, img_name)
    img.save(out_path, exif=piexif.dump(exif_dict))
    print(f"📸 {img_name} → EXIF dasar lengkap (GPS tanpa altitude + yaw + JSON)")

    # === Inject XMP pose (universal) ===
    exiftool_args = ["exiftool", "-overwrite_original"]
    if not pd.isna(yaw_deg):
        exiftool_args.append(f"-XMP:PoseYawDegrees={yaw_deg}")
    if not pd.isna(pitch_deg):
        exiftool_args.append(f"-XMP:PosePitchDegrees={pitch_deg}")
    if not pd.isna(roll_deg):
        exiftool_args.append(f"-XMP:PoseRollDegrees={roll_deg}")
    exiftool_args.append(out_path)
    run(exiftool_args)

    print(f"✅ {img_name} → XMP PoseYaw/Pitch/Roll ditambahkan\n")

print("🎉 Selesai! Semua gambar sudah punya EXIF + XMP lengkap tanpa altitude.")
